In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DEPLoss(nn.Module):
    def __init__(self, target_variance=1.0, lambda_base=1.0):
        super(DEPLoss, self).__init__()
        # Hedeflenen ideal varyans değeri ve taban lambda çarpanımız
        self.target_variance = target_variance
        self.lambda_base = lambda_base

    def forward(self, z1, z2, motion_score):
        """
        z1, z2: Modelin ürettiği gizli uzay (latent space) vektörleri
        motion_score: 0.0 (tamamen statik) ile 1.0 (çok hareketli) arasında bir değer
        """
        # 1. Benzerlik Kaybı (Similarity Loss) - Aynı girdinin temsillerini yakınlaştır
        sim_loss = F.mse_loss(z1, z2)

        # 2. Varyans Kaybı (Variance Loss) - Çöküşü engellemek için gereken standart ölçüm
        var_z1 = torch.var(z1, dim=0).mean()
        var_z2 = torch.var(z2, dim=0).mean()
        
        # Varyans hedefin altında kalırsa cezalandır (Hinge Loss mantığı)
        var_loss = F.relu(self.target_variance - var_z1) + F.relu(self.target_variance - var_z2)

        # 3. İŞTE BİZİM ÇIĞIR AÇAN KISMIMIZ: Dinamik Adaptasyon (DEP)
        lambda_dynamic = self.lambda_base * motion_score

        # Toplam Kayıp: Durağan anlarda lambda_dynamic 0'a yaklaşır, halüsinasyon engellenir.
        total_loss = sim_loss + (lambda_dynamic * var_loss)
        
        return total_loss

In [2]:
import torch
import torch.nn as nn

# Temel bir Evrişim Bloğu (YOLO / CSP mantığından ilham alınmıştır)
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super(ConvBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.silu = nn.SiLU() # ReLU yerine modern SiLU aktivasyonu

    def forward(self, x):
        return self.silu(self.bn(self.conv(x)))

# Sistemin Ana Kodlayıcısı (Edge Encoder)
class ALMP_Encoder(nn.Module):
    def __init__(self, latent_dim=128):
        super(ALMP_Encoder, self).__init__()
        
        # Görüntüyü agresif bir şekilde küçültüp özellik haritalarını (feature maps) çıkarıyoruz
        self.layer1 = ConvBlock(3, 32, stride=2)   # 224x224 -> 112x112
        self.layer2 = ConvBlock(32, 64, stride=2)  # 112x112 -> 56x56
        self.layer3 = ConvBlock(64, 128, stride=2) # 56x56 -> 28x28
        self.layer4 = ConvBlock(128, 256, stride=2)# 28x28 -> 14x14
        
        # Son özellikleri toplamak için Global Average Pooling
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # TLP (3D Arayüz) ve DEPLoss için 128 boyutlu gizli uzaya (latent space) izdüşüm
        self.projector = nn.Sequential(
            nn.Linear(256, 256),
            nn.SiLU(),
            nn.Linear(256, latent_dim)
        )

    def forward(self, x):
        # Uzamsal özellikleri çıkar
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        # Havuzlama (Pooling) ve Vektöre Çevirme
        x = self.pool(x)
        x = torch.flatten(x, 1)
        
        # Gizli Uzay (Latent Space) Vektörü -> Z
        z = self.projector(x)
        return z

# Mimariyi Test Edelim
if __name__ == "__main__":
    model = ALMP_Encoder(latent_dim=128)
    dummy_image = torch.randn(1, 3, 224, 224) # 1 adet 224x224 RGB resim simülasyonu
    z_vector = model(dummy_image)
    
    print(f"Girdi Boyutu: {dummy_image.shape}")
    print(f"Çıktı (Gizli Uzay) Boyutu: {z_vector.shape} -> Bu vektör Frontend'e gidecek!")

Girdi Boyutu: torch.Size([1, 3, 224, 224])
Çıktı (Gizli Uzay) Boyutu: torch.Size([1, 128]) -> Bu vektör Frontend'e gidecek!


In [3]:
import torch
import torch.optim as optim

# Hareket Skorunu hesaplayan özgün fonksiyonumuz
def compute_motion_score(frame1, frame2, threshold=0.05):
    """
    Squeezes the structural difference between two frames between 0.0 and 1.0.
    """
    mae_diff = torch.abs(frame1 - frame2).mean()
    motion_score = torch.sigmoid((mae_diff - threshold) * 50) 
    return motion_score.item()

# 1. Initialize Model, Loss (Criterion), and Optimizer
model = ALMP_Encoder(latent_dim=128)
criterion = DEPLoss(target_variance=1.0, lambda_base=2.0)
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("--- ALMP System Test Starting ---")

# ÇÖZÜM BURADA: Batch size (en baştaki sayı) artık 1 değil, 4. 
# Yani modele her adımda 4 adet resim gönderiyoruz.
frame_t1 = torch.randn(4, 3, 224, 224) 
frame_t2 = torch.randn(4, 3, 224, 224) 

frame_static_1 = torch.randn(4, 3, 224, 224)
frame_static_2 = frame_static_1.clone() 

def train_step(f1, f2, scenario_name):
    optimizer.zero_grad()
    
    # A. Extract Latent Vectors (128-dimensional nodes)
    z1 = model(f1)
    z2 = model(f2)
    
    # B. Compute Dynamic Motion Score
    m_score = compute_motion_score(f1, f2)
    
    # C. Apply Custom ALMP Loss (DEP)
    loss = criterion(z1, z2, motion_score=m_score)
    
    # Backpropagation
    loss.backward()
    optimizer.step()
    
    # İngilizce log çıktısı
    print(f"[{scenario_name}] Motion Score: {m_score:.4f} | Total Loss: {loss.item():.4f}")

# Run Tests
train_step(frame_t1, frame_t2, "Motion Scenario")
train_step(frame_static_1, frame_static_2, "Static Scenario")

--- ALMP System Test Starting ---
[Motion Scenario] Motion Score: 1.0000 | Total Loss: 3.9999
[Static Scenario] Motion Score: 0.0759 | Total Loss: 0.3034
